#ML MODEL WORKFLOW FOR NYC TAXI TRIP

Importing libraries

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

Loading and Sampling the Data

In [0]:
df = spark.table("silver_taxi").toPandas()
df = df.sample(n=5000, random_state=42)
df.head()
df.dtypes

Defining Feature Set

In [0]:
features = ["trip_distance", "trip_duration_minutes"]
#I'm deliberately leaving pickup_zip out for this first pass — it's categorical, not continuous, and one-hot encoding it adds a real extra step that's easy to bolt on later once you have a working baseline.
target = "fare_amount"

X = df[features]
y = df[target]

print(X.isnull().sum())
print(y.isnull().sum())

Train-Test-Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

First MLflow-Tracked Training Run

In [0]:
with mlflow.start_run(run_name="linear_regression_baseline"):
    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features", features)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(model, name="model",input_example=X_train.head(5))

    print(f"RMSE: {rmse:.2f}, R2: {r2:.3f}")

Training the Random Forest

In [0]:
with mlflow.start_run(run_name="random_forest_baseline"):
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("features", features)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(5))

    print(f"RMSE: {rmse:.2f}, R2: {r2:.3f}")

Loading the Registered Model and Running Batch Inference

In [0]:
import mlflow

model_uri = "models:/workspace.default.taxi_fare_predictor/1"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Score a batch of data (reuse your test set, or grab a fresh sample)
batch_preds = loaded_model.predict(X_test)

results = X_test.copy()
results["actual_fare"] = y_test
results["predicted_fare"] = batch_preds

results.head(10)

Model is good on average (R²=0.90) but has a real blind spot on edge cases.

Writing Predictions to a Delta Table

In [0]:
spark_df = spark.createDataFrame(results)
spark_df.write.format("delta").mode("overwrite").saveAsTable("fare_predictions")

Sanity Check on the Error Distribution

In [0]:
%sql
SELECT
  actual_fare, predicted_fare,
  round(abs(actual_fare - predicted_fare), 2) AS abs_error
FROM fare_predictions
ORDER BY abs_error DESC
LIMIT 10;

In [0]:
%sql
SELECT * FROM fare_predictions WHERE actual_fare IN (55, 52);

This model handles the flat-rate pattern well by accident of correlated distance, and its worst failures are two clear data-quality outliers, not a systemic blind spot.